<p style="text-align:right"><img src="https://www.iesb.br/content/themes/iesb-chleba-themosis/assets/imagens/logo.png" width="100"/></p>

# 🧠 Laboratório 7 — Modelos de linguagem: da contagem ao agente

**Inteligência Artificial (MDC050)** · Centro Universitário IESB · Turma ENGCNM2B
**Prof. Rodrigo Gonçalves Pinto** · Aula 08 · 14/10/2026

---

## O que você vai fazer aqui

| Etapa | O que acontece | Precisa de internet? |
|---|---|---|
| 0 | Preparar o Colab (GPU) | não |
| 1 | Treinar um modelo de bigrama com as 6 frases da aula | não |
| 2 | Gerar uma frase, ver a alucinação e calcular a temperatura | não |
| 3 | Criar sua conta e seu token no Hugging Face e ver os tokens do GPT-2 | sim |
| 4 | Ver o GPT-2 prevendo a próxima palavra | sim |
| 5 | Conversar com o Qwen, o modelo que vai trabalhar no agente do trio | sim |
| Desafio | O Qwen explica a solução do A* do seu trio | sim |

## Como entregar

1. Complete os **seis TODOs**. Embaixo de cada um há uma célula de verificação que imprime ✅ ou ❌.
2. Responda as perguntas da célula **Sua análise**, no final.
3. **Antes de entregar:** `Ambiente de execução → Reiniciar sessão e executar tudo`. Tudo precisa rodar do zero, sem erro.
4. Envie o link do notebook pelo Google Sala de Aula.

> **Crie sua cópia antes de começar:** `Arquivo → Salvar uma cópia no Drive`. Edições no arquivo original não são salvas.

---
# Etapa 0 — Preparação

## Ligando a GPU (faça isso antes de executar qualquer célula)

1. No menu, clique em **Ambiente de execução → Alterar o tipo de ambiente de execução**.
2. Em **Acelerador de hardware**, escolha **T4 GPU**.
3. Clique em **Salvar**.

**E se não houver GPU disponível?** O Colab gratuito tem uma cota de GPU por dia. Se ela acabar, escolha **CPU** e siga normalmente: **o notebook funciona igual**, só que as respostas do Qwen demoram mais (cerca de 1 minuto em vez de alguns segundos).

Execute a célula abaixo para saber em qual dos dois você está.

In [ ]:
import torch

if torch.cuda.is_available():
    dispositivo = "cuda"
    print("✅ GPU ligada. As respostas vão ser rápidas.")
else:
    dispositivo = "cpu"
    print("⚠️ Sem GPU. O notebook funciona, mas a Etapa 5 vai ser mais lenta.")

---
# Etapa 1 — Um modelo de bigrama feito à mão

## O corpus

São as mesmas seis frases da aula. Cada uma descreve uma estrada que **existe** no mapa da Romênia.

Um **modelo de bigrama** prevê a próxima palavra olhando **só a palavra anterior**. Treinar o modelo é contar: para cada palavra, quais palavras vieram logo depois dela e quantas vezes.

In [ ]:
corpus = [
    "o agente vai de arad a sibiu",
    "o agente vai de sibiu a fagaras",
    "o agente vai de sibiu a rimnicu",
    "o agente vai de rimnicu a pitesti",
    "o agente vai de pitesti a bucareste",
    "o agente vai de fagaras a bucareste",
]

# .split() separa a frase nos espaços e devolve uma lista de palavras
palavras = corpus[0].split()
print(palavras)

## Percorrendo os pares de palavras

Para cada posição `i`, a palavra **anterior** é `palavras[i]` e a **próxima** é `palavras[i + 1]`. Veja na primeira frase:

In [ ]:
palavras = corpus[0].split()

for i in range(len(palavras) - 1):
    anterior = palavras[i]
    proxima = palavras[i + 1]
    print("depois de", anterior, "veio", proxima)

## 🛠️ TODO 1 — Contar os pares

A contagem fica guardada em um **dicionário de dicionários**: `contagem["a"]["bucareste"]` diz quantas vezes `bucareste` veio depois de `a`.

O código abaixo já percorre todas as frases e já cria os dicionários vazios. **Falta uma linha:** somar 1 na contagem do par `(anterior, proxima)`.

*Dica:* `contagem[anterior][proxima] = contagem[anterior][proxima] + 1`

In [ ]:
contagem = {}

for frase in corpus:
    palavras = frase.split()
    for i in range(len(palavras) - 1):
        anterior = palavras[i]
        proxima = palavras[i + 1]

        # se é a primeira vez que vemos esta palavra, criamos o dicionário dela
        if anterior not in contagem:
            contagem[anterior] = {}
        # se é a primeira vez que vemos este par, começamos a contagem em zero
        if proxima not in contagem[anterior]:
            contagem[anterior][proxima] = 0

        # TODO 1: some 1 na contagem do par (anterior, proxima)
        pass   # apague esta linha e escreva a sua

print("Depois de 'de':", contagem["de"])
print("Depois de 'a' :", contagem["a"])

In [ ]:
# --- Verificação do TODO 1 (não altere) ---
if contagem["de"] == {"arad": 1, "sibiu": 2, "rimnicu": 1, "pitesti": 1, "fagaras": 1} and contagem["a"]["bucareste"] == 2:
    print("✅ TODO 1 correto. As contagens batem com as do quadro.")
else:
    print("❌ Ainda não. Depois de 'de' o esperado é arad 1, sibiu 2, rimnicu 1, pitesti 1, fagaras 1.")

---
# Etapa 2 — Probabilidades, geração de texto e temperatura

## 🛠️ TODO 2 — De contagem para probabilidade

A probabilidade de uma palavra é a contagem dela dividida pelo total:

`P(próxima | anterior) = contagem do par ÷ total de vezes que a anterior apareceu`

Complete a função. O total já está calculado.

In [ ]:
def probabilidade(anterior, proxima):
    total = 0
    for valor in contagem[anterior].values():
        total = total + valor

    # TODO 2: devolva a contagem do par dividida pelo total
    return None   # apague esta linha e escreva a sua


print("P(bucareste | a) =", probabilidade("a", "bucareste"))
print("P(sibiu | de)    =", probabilidade("de", "sibiu"))

In [ ]:
# --- Verificação do TODO 2 (não altere) ---
p1 = probabilidade("a", "bucareste")
p2 = probabilidade("de", "arad")
if p1 is not None and round(p1, 3) == 0.333 and round(p2, 3) == 0.167:
    print("✅ TODO 2 correto. P(bucareste | a) = 2 ÷ 6 = 0,333")
else:
    print("❌ Ainda não. P(bucareste | a) deveria ser 2 ÷ 6 = 0,333 e você obteve", p1)

## Gerando uma frase

A função abaixo escolhe a palavra **mais provável** depois de uma palavra. Ela já está pronta: leia os comentários.

In [ ]:
def mais_provavel(anterior):
    melhor_palavra = None
    melhor_contagem = 0
    for palavra in contagem[anterior]:
        if contagem[anterior][palavra] > melhor_contagem:
            melhor_palavra = palavra
            melhor_contagem = contagem[anterior][palavra]
    return melhor_palavra

print("Depois de 'de' vem:", mais_provavel("de"))
print("Depois de 'a'  vem:", mais_provavel("a"))

## 🛠️ TODO 3 — Escrever uma frase

Começando por `"o"`, repita 6 vezes: pegue a **última palavra** da frase, escolha a mais provável depois dela e **acrescente** na frase.

Falta uma linha: acrescentar `escolhida` na lista `frase`. *Dica:* `frase.append(escolhida)`

In [ ]:
frase = ["o"]

for passo in range(6):
    ultima = frase[-1]                 # [-1] é o último elemento da lista
    escolhida = mais_provavel(ultima)
    # TODO 3: acrescente a palavra escolhida na frase
    pass   # apague esta linha e escreva a sua

texto_gerado = " ".join(frase)         # junta a lista em um texto, com espaços
print(texto_gerado)

In [ ]:
# --- Verificação do TODO 3 (não altere) ---
if texto_gerado == "o agente vai de sibiu a bucareste":
    print("✅ TODO 3 correto. O modelo escreveu:", texto_gerado)
    print()
    print("Agora volte ao mapa: existe estrada entre Sibiu e Bucareste?")
    print("Não existe. Essa frase não está no corpus: o modelo colou pedaços de frases diferentes.")
    print("Isso é uma ALUCINAÇÃO: um texto fluente, plausível e falso.")
else:
    print("❌ Ainda não. O esperado é 'o agente vai de sibiu a bucareste' e saiu:", texto_gerado)

## 🛠️ TODO 4 — Softmax com temperatura

Um modelo de verdade não devolve probabilidades: devolve **pontuações**. Para o contexto *"o agente vai de sibiu a"*, usamos na aula:

| fagaras | rimnicu | bucareste | arad |
|---|---|---|---|
| 2,0 | 1,5 | 0,5 | −1,0 |

Para virar probabilidade (softmax com temperatura):
1. divida cada pontuação pela temperatura;
2. calcule `math.exp` de cada resultado;
3. divida cada exponencial pela soma de todas.

Os passos 1 e 3 já estão prontos. **Complete o passo 2.**

In [ ]:
import math

def softmax(pontuacoes, temperatura):
    # passo 1: dividir pela temperatura
    divididas = []
    for p in pontuacoes:
        divididas.append(p / temperatura)

    # passo 2: TODO 4 — calcule math.exp de cada valor de 'divididas'
    exponenciais = []
    for d in divididas:
        pass   # apague esta linha e escreva: exponenciais.append(math.exp(d))

    # passo 3: dividir cada exponencial pela soma
    soma = sum(exponenciais)
    probabilidades = []
    for e in exponenciais:
        probabilidades.append(e / soma)
    return probabilidades


nomes = ["fagaras", "rimnicu", "bucareste", "arad"]
pontuacoes = [2.0, 1.5, 0.5, -1.0]

for temperatura in [0.5, 1.0, 2.0]:
    probs = softmax(pontuacoes, temperatura)
    print("Temperatura", temperatura)
    for i in range(len(nomes)):
        if len(probs) > i:
            print("   ", nomes[i], round(probs[i] * 100, 1), "%")

In [ ]:
# --- Verificação do TODO 4 (não altere) ---
p = softmax(pontuacoes, 1.0)
if len(p) == 4 and round(p[0], 3) == 0.532 and round(softmax(pontuacoes, 0.5)[0], 3) == 0.704:
    print("✅ TODO 4 correto. Os números batem com os da aula.")
    print("   Repare: com temperatura 2,0 a estrada falsa (bucareste) sobe para", round(softmax(pontuacoes, 2.0)[2] * 100, 1), "%")
else:
    print("❌ Ainda não. Com temperatura 1,0, fagaras deveria dar 53,2 %.")

---
# Etapa 3 — Hugging Face: conta, token e os tokens do GPT-2

O **Hugging Face** (huggingface.co) é um repositório público de modelos abertos. Hoje vamos baixar dois modelos de lá: o **GPT-2** e o **Qwen**.

## Passo a passo: criar o seu token de acesso (faça uma vez só)

**1. Crie a conta (se ainda não tiver)**
- Acesse **huggingface.co** e clique em **Sign Up**.
- Use seu e-mail, crie uma senha e **confirme o e-mail** pelo link que chega na sua caixa de entrada.

**2. Crie o token**
- Com a conta aberta, clique na sua foto (canto superior direito) → **Settings** → **Access Tokens**.
- Clique em **Create new token**.
- Em **Token type**, escolha **Read** (somente leitura: basta para baixar modelos).
- Dê um nome, por exemplo `colab-ia`, e clique em **Create token**.
- **Copie o token** (começa com `hf_`). Ele só aparece inteiro uma vez; se perder, crie outro.

**3. Guarde o token no Colab (nunca no código)**
- Na barra lateral esquerda do Colab, clique no ícone da **chave 🔑 (Secrets / Segredos)**.
- Clique em **Adicionar novo secret**.
- Em **Nome**, escreva exatamente `HF_TOKEN`. Em **Valor**, cole o token.
- Ative a chave **Acesso ao notebook**.

> **Por que tanto cuidado?** Este notebook será compartilhado com o professor e, no projeto, com os colegas. Um token escrito no código pode ser copiado por qualquer pessoa. Nos Segredos, ele fica só na sua conta. O Projeto Acadêmico exige isso.

> **E se eu não conseguir criar o token agora?** Os dois modelos de hoje são abertos e baixam mesmo sem login. Pule a próxima célula e siga; o token evita avisos e limites quando a turma inteira baixa ao mesmo tempo.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")   # lê o token dos Segredos do Colab
login(token=token)
print("✅ Login no Hugging Face feito.")

## O tokenizador do GPT-2

O modelo não lê palavras: lê **tokens**, pedaços de palavras. A célula abaixo baixa só o **tokenizador** do GPT-2 (alguns megabytes) e mostra como ele corta um texto.

In [ ]:
from transformers import AutoTokenizer

tokenizador_gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2")

def mostrar_tokens(texto):
    ids = tokenizador_gpt2.encode(texto)
    pedacos = []
    for numero in ids:
        pedacos.append(tokenizador_gpt2.decode([numero]))
    print(len(ids), "tokens:", pedacos)

mostrar_tokens("Quero ir de Arad até Bucareste pelo caminho mais curto.")
mostrar_tokens("I want to go from Arad to Bucharest by the shortest route.")

## 🛠️ TODO 5 — Os nomes do domínio do seu trio

Escreva na lista abaixo **pelo menos cinco nomes** do domínio do projeto do seu trio: estados, lugares, ações, objetos. Por exemplo, se o trio trabalha com rotas no DF: `"Taguatinga"`, `"Rodoviária do Plano Piloto"`, `"Eixo Monumental"`…

Depois execute e veja em quantos pedaços cada nome é cortado.

In [ ]:
# TODO 5: troque os exemplos pelos nomes do domínio do SEU trio (pelo menos cinco)
nomes_do_dominio = ["exemplo 1", "exemplo 2"]

for nome in nomes_do_dominio:
    print(nome)
    mostrar_tokens(" " + nome)    # o espaço antes imita o nome no meio de uma frase
    print()

In [ ]:
# --- Verificação do TODO 5 (não altere) ---
if len(nomes_do_dominio) >= 5 and "exemplo 1" not in nomes_do_dominio:
    print("✅ TODO 5 correto.", len(nomes_do_dominio), "nomes do domínio do trio.")
    print("   Guarde esta lista: ela vai voltar nas próximas aulas.")
else:
    print("❌ Ainda não. Escreva pelo menos cinco nomes do domínio do SEU trio.")

---
# Etapa 4 — O GPT-2 prevendo a próxima palavra

Agora baixamos o modelo inteiro (cerca de 500 MB). O GPT-2 é um **modelo base**: foi treinado só para prever a próxima palavra, quase sempre em inglês.

A função `proximas_palavras` já está pronta: ela mostra as cinco palavras mais prováveis, com a probabilidade **real** calculada pelo modelo. É a mesma softmax do TODO 4, só que sobre 50.257 tokens.

In [ ]:
from transformers import AutoModelForCausalLM

gpt2 = AutoModelForCausalLM.from_pretrained("openai-community/gpt2").to(dispositivo)
gpt2.eval()
print("✅ GPT-2 carregado.")


def proximas_palavras(texto, quantas=5):
    ids = tokenizador_gpt2(texto, return_tensors="pt")["input_ids"].to(dispositivo)
    with torch.no_grad():
        pontuacoes = gpt2(ids).logits[0, -1]          # pontuações da próxima palavra
    probs = torch.softmax(pontuacoes, dim=-1)         # pontuações -> probabilidades
    melhores = torch.topk(probs, quantas)
    print("Depois de:", repr(texto))
    for i in range(quantas):
        numero = int(melhores.indices[i])
        print("   ", repr(tokenizador_gpt2.decode([numero])), "->", round(float(melhores.values[i]) * 100, 1), "%")
    print()


proximas_palavras("The capital of France is")
proximas_palavras("The agent drives from Arad to")
proximas_palavras("O agente vai de Sibiu a")

**Observe:** em inglês, as previsões fazem sentido. Em português, o GPT-2 se perde: ele viu pouquíssimo português no treino.

## Gerando texto com temperaturas diferentes

A função `continuar_gpt2` gera 20 tokens sorteando a próxima palavra de acordo com as probabilidades. Rode a célula e compare as três temperaturas.

In [ ]:
def continuar_gpt2(texto, temperatura):
    torch.manual_seed(42)                                     # sorteio reprodutível
    entrada = tokenizador_gpt2(texto, return_tensors="pt").to(dispositivo)
    saida = gpt2.generate(**entrada, max_new_tokens=20, do_sample=True,
                          temperature=temperatura, pad_token_id=tokenizador_gpt2.eos_token_id)
    return tokenizador_gpt2.decode(saida[0])


for temperatura in [0.2, 1.0, 2.0]:
    print("Temperatura", temperatura, ":")
    print("   ", continuar_gpt2("The shortest route from Arad to Bucharest is", temperatura))
    print()

---
# Etapa 5 — O Qwen: o modelo que vai trabalhar no agente

O **Qwen2.5-1.5B-Instruct** é um **modelo instruído**: além de prever a próxima palavra, ele recebeu um treino extra com instruções e conversas. Por isso entende português, segue ordens e devolve dados no formato que pedimos.

| | GPU T4 | Sem GPU |
|---|---|---|
| Baixar e carregar (3 GB) | cerca de 1 min | cerca de 1 min e 15 s |
| Uma resposta curta | alguns segundos | cerca de 1 min |

A célula abaixo baixa e carrega o modelo. **Ela demora: espere terminar.**

In [ ]:
nome_qwen = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizador_qwen = AutoTokenizer.from_pretrained(nome_qwen)

if dispositivo == "cuda":
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float16).to(dispositivo)
else:
    qwen = AutoModelForCausalLM.from_pretrained(nome_qwen, dtype=torch.float32)
qwen.eval()

print("✅ Qwen carregado em", dispositivo)

## A função `conversar`

Ela já está pronta. Você só precisa entender **o que entra e o que sai**:

- **Entra** uma lista de mensagens. Cada mensagem é um dicionário com o papel (`role`) e o texto (`content`):
  - `"system"`: as regras gerais que o modelo deve seguir;
  - `"user"`: o pedido do usuário.
- **Sai** o texto da resposta.

In [ ]:
def conversar(mensagens, max_tokens=150):
    entrada = tokenizador_qwen.apply_chat_template(
        mensagens, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(dispositivo)
    with torch.no_grad():
        saida = qwen.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    tamanho_da_pergunta = entrada["input_ids"].shape[-1]
    return tokenizador_qwen.decode(saida[0][tamanho_da_pergunta:], skip_special_tokens=True)


mensagens = [
    {"role": "system", "content": "Responda em português, em no máximo duas frases."},
    {"role": "user", "content": "O que é o algoritmo A*?"},
]
print(conversar(mensagens))

## A mesma pergunta para os dois modelos

Vamos fazer a mesma pergunta ao GPT-2 (base) e ao Qwen (instruído). Compare as respostas.

In [ ]:
pergunta = "Qual é a capital do Brasil?"

print("GPT-2 (base):")
print("   ", continuar_gpt2(pergunta, 0.7))
print()
print("Qwen (instruído):")
print("   ", conversar([{"role": "user", "content": pergunta}], max_tokens=40))

**O que aconteceu:** o GPT-2 não *responde*, ele *continua* o texto, como se a pergunta fosse o começo de uma página da internet. O Qwen responde, porque foi treinado com conversas. Os dois fazem a mesma coisa por dentro: prever o próximo token.

---
# Desafio Final (Entrega Obrigatória) — O Qwen explica a solução do A* do trio

No agente do Marco 2, o **A* calcula** e o **modelo explica**. O modelo não sabe nada que não esteja na mensagem: por isso a solução do A* precisa ir **dentro** do pedido.

## 🛠️ TODO 6 — Monte o pedido com a solução do seu A*

1. Troque os valores de `pedido`, `caminho` e `custo` pela solução do **A* do seu trio** (do Marco 1). Se o A* ainda não estiver pronto, use um resultado calculado à mão.
2. Complete o texto da instrução `regras`: peça ao modelo para explicar a solução **em português**, **em até quatro frases**, e para **usar apenas os dados recebidos, sem inventar** nada.

In [ ]:
# 1) a solução do A* do seu trio (troque o exemplo da Romênia)
pedido = "Quero ir de Arad a Bucareste pelo caminho mais curto."
caminho = ["Arad", "Sibiu", "Rimnicu Vilcea", "Pitesti", "Bucareste"]
custo = 418

# 2) TODO 6: escreva as regras para o modelo
regras = ""

# o pedido para o modelo junta tudo em um texto só
dados = "Pedido do usuário: " + pedido + "\n"
dados = dados + "Caminho encontrado pelo A*: " + " -> ".join(caminho) + "\n"
dados = dados + "Custo total: " + str(custo)

mensagens = [
    {"role": "system", "content": regras},
    {"role": "user", "content": dados},
]
print("O que vai para o modelo:")
print(dados)

In [ ]:
# --- Verificação do TODO 6 (não altere) ---
texto_regras = regras.lower()
pede_portugues = "portugu" in texto_regras
pede_nao_inventar = "invent" in texto_regras or "apenas" in texto_regras or "somente" in texto_regras
if pede_portugues and pede_nao_inventar and len(caminho) >= 2:
    print("✅ TODO 6 correto. As regras pedem português e proíbem inventar.")
else:
    print("❌ Ainda não. As regras precisam pedir resposta em português E proibir o modelo de inventar dados.")

In [ ]:
# Agora o Qwen explica. Na CPU, esta célula leva cerca de 1 minuto.
explicacao = conversar(mensagens)
print(explicacao)

### Sua análise — edite esta célula

Responda em 4 a 6 linhas:

1. **Os dados:** a explicação usou exatamente o caminho e o custo que vocês passaram? O modelo acrescentou algum número, lugar ou passo que **não** estava na mensagem?
2. **Os tokens:** dos nomes do domínio do trio (TODO 5), qual foi cortado em mais pedaços? Por que isso pode atrapalhar o agente?
3. **A decisão de projeto:** com base na frase *"o agente vai de sibiu a bucareste"*, explique por que o agente do trio usa o A* para calcular a solução em vez de pedir a solução ao modelo.

---

_Escreva sua análise aqui, substituindo esta linha._

---

---
## ✅ Checklist antes de entregar

- [ ] Os seis TODOs com ✅ na verificação.
- [ ] O token do Hugging Face está nos **Segredos** (🔑) e **não aparece** em nenhuma célula.
- [ ] A solução do A* do **seu trio** foi usada no Desafio Final.
- [ ] A célula **Sua análise** foi preenchida.
- [ ] `Ambiente de execução → Reiniciar sessão e executar tudo` roda sem erro.

## Resumo de fixação

| Conceito da aula | Onde apareceu no notebook |
|---|---|
| Treinar um modelo de linguagem | Contar pares de palavras (TODO 1) |
| Probabilidade da próxima palavra | Contagem ÷ total (TODO 2) |
| Alucinação | "o agente vai de sibiu a bucareste" (TODO 3) |
| Temperatura | Softmax com temperatura (TODO 4) e `continuar_gpt2` |
| Tokens | `mostrar_tokens` e os nomes do domínio (TODO 5) |
| Modelo base × instruído | GPT-2 continua o texto; Qwen responde |
| O modelo só sabe o que recebe | A solução do A* dentro da mensagem (TODO 6) |

---
**Para a próxima aula (21/10 — Prompt e saída estruturada):** traga a função **A* do Marco 1 funcionando no Colab**, recebendo o estado inicial e o objetivo como parâmetros e devolvendo o caminho e o custo.